In [1]:
import pandas as pd

# ── 교재 코드: Heap ──
class Heap:
    def __init__(self, *args):
        if len(args) != 0:
            self.__A = args[0]
        else:
            self.__A = []

    def insert(self, x):
        self.__A.append(x)
        self.__percolateUp(len(self.__A) - 1)

    def __percolateUp(self, i: int):
        if i > 0:
            parent = (i - 1) // 2
            if self.__A[i] > self.__A[parent]:
                self.__A[i], self.__A[parent] = self.__A[parent], self.__A[i]
                self.__percolateUp(parent)

    def deleteMax(self):
        if not self.isEmpty():
            max_val = self.__A[0]
            self.__A[0] = self.__A.pop()
            self.__percolateDown(0)
            return max_val
        else:
            return None

    def __percolateDown(self, i: int):
        child = 2 * i + 1
        right = 2 * i + 2
        if child <= len(self.__A) - 1:
            if right <= len(self.__A) - 1 and self.__A[child] < self.__A[right]:
                child = right
            if self.__A[i] < self.__A[child]:
                self.__A[i], self.__A[child] = self.__A[child], self.__A[i]
                self.__percolateDown(child)

    def max(self):
        return self.__A[0]

    def buildHeap(self):
        for i in range((len(self.__A) - 2) // 2, -1, -1):
            self.__percolateDown(i)

    def isEmpty(self) -> bool:
        return len(self.__A) == 0

    def clear(self):
        self.__A = []

    def size(self) -> int:
        return len(self.__A)


# ── 교재 코드: BidirectNode + CircularDoublyLinkedList ──
class BidirectNode:
    def __init__(self, item, prev=None, next=None):
        self.item = item
        self.prev = prev
        self.next = next

class CircularDoublyLinkedList:
    def __init__(self):
        self.__head = BidirectNode("dummy", None, None)
        self.__head.prev = self.__head
        self.__head.next = self.__head
        self.__numItems = 0

    def append(self, newItem) -> None:
        prev = self.__head.prev
        newNode = BidirectNode(newItem, prev, self.__head)
        prev.next = newNode
        self.__head.prev = newNode
        self.__numItems += 1

    def isEmpty(self) -> bool:
        return self.__numItems == 0

    def size(self) -> int:
        return self.__numItems

    def getNode(self, i: int) -> BidirectNode:
        curr = self.__head
        for _ in range(i + 1):
            curr = curr.next
        return curr

    def __iter__(self):
        return CircularDoublyLinkedListIterator(self)

class CircularDoublyLinkedListIterator:
    def __init__(self, alist):
        self.__head = alist.getNode(-1)
        self.iterPosition = self.__head.next

    def __next__(self):
        if self.iterPosition == self.__head:
            raise StopIteration
        item = self.iterPosition.item
        self.iterPosition = self.iterPosition.next
        return item

    def __iter__(self):
        return self


# ════════════════════════════════════════════════
# 과제 1. 힙으로 저장 → 생일이 느린 순서로 10명 출력
# ════════════════════════════════════════════════
CSV_PATH = "https://raw.githubusercontent.com/go-butter/DS_assignments/main/birthday.csv"
df = pd.read_csv(CSV_PATH)

name_col  = df.columns[0]
year_col  = df.columns[1]
month_col = df.columns[2]
day_col   = df.columns[3]

h = Heap()

for _, row in df.iterrows():
    name = row[name_col]
    if pd.isna(row[year_col]) or pd.isna(row[month_col]) or pd.isna(row[day_col]):
        print(f"[예외처리] {name}: 생년월일 정보 없음 → 힙 삽입 제외")
        continue
    year  = int(row[year_col])
    month = int(row[month_col])
    day   = int(row[day_col])
    bday_int = year * 10000 + month * 100 + day
    h.insert((bday_int, name))

print()
print("생일이 느린 순서 Top 10")
print("=" * 30)
for rank in range(1, 11):
    item = h.deleteMax()
    if item:
        bday, name = item
        y = bday // 10000
        m = (bday % 10000) // 100
        d = bday % 100
        print(f"{rank:2d}. {name:<12s}  {y}년 {m:02d}월 {d:02d}일")


# ════════════════════════════════════════════════
# 과제 2. CircularDoublyLinkedList → 같은 조 출력
# ════════════════════════════════════════════════
MY_GROUP = ['이시현', '이예지', '이유민', '이윤서', '이주은',
            '이채영', '장채은', '정희원', '조은서', '진가연', '최윤지']

cdll = CircularDoublyLinkedList()

for _, row in df.iterrows():
    name = row[name_col]
    if name not in MY_GROUP:  # 같은 조 아니면 스킵
        continue
    if pd.isna(row[year_col]) or pd.isna(row[month_col]) or pd.isna(row[day_col]):
        cdll.append((name, None))
    else:
        y = int(row[year_col])
        m = int(row[month_col])
        d = int(row[day_col])
        cdll.append((name, f"{y}년 {m:02d}월 {d:02d}일"))

print()
print(f"같은 조 친구들 (총 {len(MY_GROUP)}명)")
print("=" * 35)
for name, bday in cdll:
    if bday is None:
        print(f"  {name:<12s}  생년월일 정보 없음")
    else:
        print(f"  {name:<12s}  {bday}")

[예외처리] 유지민: 생년월일 정보 없음 → 힙 삽입 제외

생일이 느린 순서 Top 10
 1. 이윤서           2006년 12월 27일
 2. 정희원           2006년 12월 21일
 3. 김효린           2006년 12월 16일
 4. 이예은           2006년 12월 09일
 5. 김주영           2006년 11월 20일
 6. 전은빈           2006년 11월 07일
 7. 이하연           2006년 09월 22일
 8. 김우현           2006년 09월 01일
 9. 최윤지           2006년 08월 30일
10. 유가현           2006년 08월 22일

같은 조 친구들 (총 11명)
  이시현           2006년 01월 25일
  이예지           2005년 01월 04일
  이유민           2005년 07월 11일
  이윤서           2006년 12월 27일
  이주은           2006년 07월 05일
  이채영           2006년 04월 18일
  장채은           2005년 05월 08일
  정희원           2006년 12월 21일
  조은서           2005년 07월 19일
  진가연           2005년 11월 12일
  최윤지           2006년 08월 30일


#3. buildHeap에서 넘어가는 원소 수
`buildHeap()`은 인덱스 `(n-2)//2` 부터 `0`까지만 `percolateDown`을 수행한다.
리프 노드는 자식이 없으므로 스며내리기를 할 필요가 없어 그냥 넘어간다.
넘어가는 원소의 수는 **⌈n/2⌉** 개이다.

In [2]:
import math
print("(n, percolateDown 대상, 넘어가는 원소)")
for n in [7, 10, 15, 16]:
    check = (n - 2) // 2 + 1
    skip  = math.ceil(n / 2)
    print(f"(n={n}, {check}개, {skip}개)")

(n, percolateDown 대상, 넘어가는 원소)
(n=7, 3개, 4개)
(n=10, 5개, 5개)
(n=15, 7개, 8개)
(n=16, 8개, 8개)


(결과) (n, percolateDown 대상, 넘어가는 원소)
(n=7, 3개, 4개)
(n=10, 5개, 5개)
(n=15, 7개, 8개)
(n=16, 8개, 8개)

# 4. percolateDown의 최악/최선 시간 복잡도

- **최악의 경우**: 루트에서 리프까지 끝까지 내려가야 할 때 → **Θ(log n)**
- **최선의 경우**: 첫 번째 비교에서 바로 멈출 때 → **Θ(1)**

In [4]:
class HeapWithCounter:
    def __init__(self, A):
        self.__A = A[:]
        self.count = 0

    def __percolateDown(self, i):
        child = 2 * i + 1
        right = 2 * i + 2
        if child <= len(self.__A) - 1:
            if right <= len(self.__A) - 1 and self.__A[child] < self.__A[right]:
                child = right
            self.count += 1
            if self.__A[i] < self.__A[child]:
                self.__A[i], self.__A[child] = self.__A[child], self.__A[i]
                self.__percolateDown(child)

    def buildHeap(self):
        for i in range((len(self.__A) - 2) // 2, -1, -1):
            self.__percolateDown(i)

worst = list(range(1, 8))
best  = [7, 5, 6, 3, 4, 1, 2]

h1 = HeapWithCounter(worst)
h1.buildHeap()
h2 = HeapWithCounter(best)
h2.buildHeap()

n = 7
print(f"n={n},  트리 높이 = log n = {int(math.log2(n))} / ")
print(f"최악 (오름차순 입력) -> 비교 횟수: {h1.count}  ->  Θ(log n) / ")
print(f"최선 (이미 힙 상태) -> 비교 횟수: {h2.count}  ->  Θ(1)")

n=7,  트리 높이 = log n = 2 / 
최악 (오름차순 입력) -> 비교 횟수: 4  ->  Θ(log n) / 
최선 (이미 힙 상태) -> 비교 횟수: 3  ->  Θ(1)


(결과) n=7,  트리 높이 = log n = 2 /
최악 (오름차순 입력) -> 비교 횟수: 4  ->  Θ(log n) /
최선 (이미 힙 상태) -> 비교 횟수: 3  ->  Θ(1)

#5. 힙의 맨 마지막 원소 삭제
힙의 마지막 원소(A[n-1])를 삭제하는 것은 간단한 일이다.
마지막 노드는 부모보다 작거나 같으므로 삭제해도 힙 속성이 깨지지 않는다.
따라서 percolateDown이나 percolateUp 같은 추가 작업이 전혀 필요 없다.

In [5]:
heap_arr = [7, 5, 6, 3, 4, 1, 2]
print(f"삭제 전:  {heap_arr}, ")
last = heap_arr.pop()
print(f"삭제된 원소: {last}, ")
print(f"삭제 후:  {heap_arr}  -> 힙 속성 유지됨")

삭제 전:  [7, 5, 6, 3, 4, 1, 2], 
삭제된 원소: 2, 
삭제 후:  [7, 5, 6, 3, 4, 1]  -> 힙 속성 유지됨


(결과) 삭제 전:  [7, 5, 6, 3, 4, 1, 2],
삭제된 원소: 2,
삭제 후:  [7, 5, 6, 3, 4, 1]  -> 힙 속성 유지됨